In [0]:
df_silver = spark.read.table("he_prod_incen_analytics_dbw_01.silver.fact_procedures")
df_dim_members = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_members")
df_dim_providers = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_providers")
df_dim_facilities = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_facilities")
df_dim_date = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_date")
df_dim_cpt = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_cpt_codes")
df_fact_claims = spark.read.table("he_prod_incen_analytics_dbw_01.gold.fact_claims") # Link to Hub!

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, coalesce, lit, col

df_gold = df_silver.withColumn("procedure_sk", monotonically_increasing_id() + 1)

# Lookups
df_gold = df_gold.join(df_fact_claims.select(col("claim_sk").alias("gold_claim_sk"), "claim_id"), on="claim_id", how="left").withColumn("gold_claim_sk", coalesce(col("gold_claim_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_members.select(col("member_sk").alias("gold_member_sk"), "member_id"), on="member_id", how="left").withColumn("gold_member_sk", coalesce(col("gold_member_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_providers.select(col("provider_sk").alias("gold_provider_sk"), "provider_id"), on="provider_id", how="left").withColumn("gold_provider_sk", coalesce(col("gold_provider_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_facilities.select(col("facility_sk").alias("gold_facility_sk"), "facility_id"), on="facility_id", how="left").withColumn("gold_facility_sk", coalesce(col("gold_facility_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_date.select(col("date_sk").alias("gold_date_sk"), col("date").alias("procedure_date")), on="procedure_date", how="left").withColumn("gold_date_sk", coalesce(col("gold_date_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_cpt.select(col("cpt_sk").alias("gold_cpt_sk"), "cpt_code"), on="cpt_code", how="left").withColumn("gold_cpt_sk", coalesce(col("gold_cpt_sk"), lit(-1)))
df_gold = df_gold.join(df_dim_providers.select(col("provider_sk").alias("gold_surgeon_sk"), col("provider_id").alias("surgeon_id")), on="surgeon_id", how="left").withColumn("gold_surgeon_sk", coalesce(col("gold_surgeon_sk"), lit(-1)))

df_gold = df_gold.select(
    "procedure_sk", "procedure_id", col("gold_claim_sk").alias("claim_sk"),
    col("gold_member_sk").alias("member_sk"), col("gold_provider_sk").alias("provider_sk"), 
    col("gold_facility_sk").alias("facility_sk"), col("gold_date_sk").alias("procedure_date_sk"),
    col("gold_cpt_sk").alias("cpt_code_sk"), col("gold_surgeon_sk").alias("surgeon_sk"),
    "total_procedure_charge", "insurance_paid", "complication", "outcome_status"
)

display(df_gold.limit(5))

In [0]:
df_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("he_prod_incen_analytics_dbw_01.gold.fact_procedures")
print("✅ Successfully wrote Fact_Procedures to Gold layer.")